# 1. EDA y Preprocesamiento — Telco Customer Churn

Este notebook hace el análisis exploratorio del dataset y deja documentadas las decisiones de limpieza que luego usará el módulo `app/preprocessing.py` durante el entrenamiento.

Objetivo: entender la estructura de los datos, detectar problemas (valores faltantes, tipos incorrectos, desbalance de clases) y proponer un esquema de preprocesamiento robusto.

Integrantes: Juan Garcia, Alejandro Carvajal, Luis Esteban Mariño

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

DATA_PATH = "../data/telco_churn.csv"
RANDOM_STATE = 42

## 1.1 Carga inicial

Leemos el CSV y revisamos las dimensiones, tipos de datos y primeras filas.

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Dimensiones:", df.shape)
df.head()

In [ ]:
df.info()

 El dataset trae aproximadamente 7043 filas y 21 columnas. La columna `customerID` es un identificador único sin valor predictivo; la quitamos. La columna `TotalCharges` aparece como `object`: pandas no la reconoció como numérica porque contiene espacios en blanco (`" "`) en clientes con `tenure==0` (recién contratados). Eso es un valor faltante disfrazado y hay que arreglarlo.

In [ ]:
df = df.drop(columns=["customerID"])

# Forzar TotalCharges a numérico; los blancos se vuelven NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("NaNs en TotalCharges tras coerción:", df["TotalCharges"].isna().sum())
df.loc[df["TotalCharges"].isna(), ["tenure", "MonthlyCharges", "TotalCharges"]].head()

**Análisis**: como esperábamos, los faltantes coinciden con `tenure == 0`. Como son pocos (~11), podemos imputarlos con la mediana en el pipeline sin afectar significativamente la distribución.

## 1.2 Variable objetivo

`Churn` viene como Yes/No. La convertimos a binaria (1 = abandona, 0 = se queda) y revisamos el balance.

In [ ]:
y = (df["Churn"] == "Yes").astype(int)

print("Distribución de churn:")
print(y.value_counts(normalize=True).round(4))

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.countplot(x=df["Churn"], order=["No", "Yes"], ax=ax)
ax.set_title("Distribución de la variable objetivo")
plt.show()

Aproximadamente el 27% de los clientes abandona, el 73% se queda. Hay desbalance moderado pero manejable. Estrategias posibles:
- Usar `class_weight='balanced'` en Random Forest / LightGBM.
- Usar `scale_pos_weight ≈ 73/27 ≈ 2.7` en XGBoost.
- Optimizar el umbral de decisión basándose en la curva precision-recall en lugar de fijar 0.5.

No vamos a hacer oversampling/SMOTE porque para árboles boosting no suele aportar valor y sí complica la interpretación.

## 1.3 Variables numéricas

Tres columnas estrictamente numéricas: `tenure`, `MonthlyCharges`, `TotalCharges`. Más `SeniorCitizen` que ya viene 0/1.

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
df[num_cols].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, num_cols):
    sns.histplot(data=df, x=col, hue="Churn", kde=True, ax=ax,
                 bins=30, element="step", common_norm=False)
    ax.set_title(f"Distribución de {col} por Churn")
plt.tight_layout()
plt.show()

**Análisis**:
- **`tenure`**: la distribución de los que abandonan está fuertemente sesgada hacia valores bajos. Los clientes nuevos (≤ 12 meses) tienen muchísima más probabilidad de irse — es la señal más obvia del dataset.
- **`MonthlyCharges`**: los churners pagan en promedio más al mes. Los planes premium o de fibra óptica generan más insatisfacción.
- **`TotalCharges`**: combina tenure × cargos mensuales, así que es algo redundante con esas dos. Aún así la dejamos porque los modelos de árboles manejan bien la correlación.

In [ ]:
# Matriz de correlación entre numéricas
corr = df[num_cols + ["SeniorCitizen"]].copy()
corr["Churn"] = y
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(corr.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlaciones (numéricas vs Churn)")
plt.show()

 `tenure` tiene correlación negativa con `Churn` (más antigüedad → menos abandono); `MonthlyCharges` correlaciona positivamente. La correlación de Pearson es lineal y subestima relaciones no lineales — los modelos de árbol capturarán el resto.

## 1.4 Variables categóricas

Quince columnas categóricas. Inspeccionamos cardinalidad y tasa de churn por categoría.

In [ ]:
cat_cols = [c for c in df.columns if df[c].dtype == object and c != "Churn"]
print("Columnas categóricas:", len(cat_cols))
for c in cat_cols:
    print(f"  {c}: {df[c].nunique()} valores → {sorted(df[c].unique())[:5]}...")

In [ ]:
# Tasa de churn por categoría — Top variables más discriminantes
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
top_cats = ["Contract", "PaymentMethod", "InternetService",
            "OnlineSecurity", "TechSupport", "PaperlessBilling",
            "Dependents", "Partner", "SeniorCitizen"]
for ax, c in zip(axes.flat, top_cats):
    rates = df.groupby(c)["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values()
    rates.plot(kind="barh", ax=ax, color="steelblue")
    ax.axvline(0.265, color="red", linestyle="--", linewidth=1, label="Tasa global")
    ax.set_title(f"Tasa de churn por {c}")
    ax.set_xlabel("P(Churn = Yes)")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Los patrones que el modelo debería capturar:
- `Contract = Month-to-month` dispara la tasa de churn por encima del 40%, mientras `Two year` la baja a menos del 5%. Variable muy poderosa.
- `PaymentMethod = Electronic check` está asociado a tasas altas, posiblemente porque correlaciona con clientes nuevos / contratos mes a mes.
- `InternetService = Fiber optic` tiene churn más alto que DSL — los clientes de fibra están menos satisfechos (¿precio? ¿calidad de servicio?). Vale la pena destacarlo al área de negocio.
- `OnlineSecurity` y `TechSupport` muestran que cuando el cliente NO tiene estos servicios contratados el churn es más alto. Hay una oportunidad clara de retención cruzando estos servicios.

## 1.5 Decisiones de preprocesamiento

Con base en el EDA, fijamos las siguientes decisiones que se implementan en `app/preprocessing.py`:

| Decisión | Razón |
|---|---|
| Eliminar `customerID` | identificador, no aporta señal |
| `TotalCharges` → numérico, NaN → mediana | corrige el dtype y rellena los faltantes de tenure=0 |
| Numéricas: `StandardScaler` | no es indispensable para árboles, pero deja el pipeline portable a otros modelos |
| Categóricas: `OneHotEncoder(handle_unknown='ignore')` | en serving puede llegar una categoría no vista; mejor ignorarla que crashear |
| `train_test_split(stratify=y, test_size=0.2)` | mantiene la proporción de churn entre train y test |
| `StratifiedKFold(5)` | evalua robustez de cada hiperparámetro respetando el balance |

Estas decisiones se cristalizan en `build_preprocessor()` y `build_feature_frame()` del módulo, que también será importado por la API en producción para garantizar el mismo tratamiento.

